# Z2 / Z2: one bounded proposal-and-check cycle

This notebook reproduces a small example of LLM-assisted lattice construction with deterministic checks. It uses **one Higgs-type and fifteen Coulomb-type components** in the all-fixed geometry, one all-Coulomb control, and four **deliberately invalid fixtures**.

The arithmetic ansatz is conditional. A scoped `PASS` does not establish a complete heterotic dual. Full Narain embedding, operator lifts and spectrum/branch matching remain `UNRESOLVED`. Run this notebook from the extracted bundle directory; the checker itself uses only Python's standard library.


In [1]:
import json
from copy import deepcopy
from pathlib import Path
from fractions import Fraction as F
import checker

ROOT = Path.cwd()
fixtures = json.loads((ROOT / "fixtures.json").read_text())
result = checker.run_suite(fixtures)
baseline = fixtures["cases"][0]
control = fixtures["cases"][1]
raw_proposal = json.loads((ROOT / "logs" / "raw_model_response.json").read_text())
proposal_case = deepcopy(baseline)
proposal_case.update({key: raw_proposal[key] for key in ("V", "delta", "N", "phi")})
proposal_case["id"] = "logged_model_proposal"
print(json.dumps(result["summary"], indent=2))
print("\nGeometric preflight: (fixed components, exchanged pairs)")
for name, data in result["geometry_preflight"].items():
    print(name, (data["fixed_components"], data["exchanged_pairs"]))


{
  "case_count": 6,
  "execution_errors": 0,
  "scoped_pass": 2,
  "scoped_fail": 4,
  "overall_unresolved": 2,
  "overall_fail": 4,
  "verified_duals": 0
}

Geometric preflight: (fixed components, exchanged pairs)
all_fixed (16, 0)
translation_i_over_2 (0, 8)
coordinate_exchange (4, 6)


## Local target and lattice order

The selected action is $(z^1,z^2)\mapsto(-z^1,z^2)$ on $T^4/\mathbb Z_2$; it fixes all sixteen singular components. The other two actions are checked only as geometric preflight cases.

For $V=(e_7+e_8)/4$, the selected root pair has phase $-1$, while the other selected pairs have phase $+1$. The **whole selected adjoint** has signs $(+,-,-)$: its Cartan generator is even. Multiplying by the scalar signs $(+,-,-)$ gives local scalar dimensions $(1,2,2)$; the fifteen untouched factors produce totals $(46,2,2)$ and vector dimension $46$.

The root action has order two, but the lattice representative has order four. The additional full-$E_8$ diagnostic illustrates why an abstract $A_1$ subsystem calculation alone is not a physical Narain spectrum.


In [2]:
evaluated = checker.evaluate_candidate(proposal_case)
by_name = {entry["name"]: entry for entry in evaluated["checks"]}
V = tuple(F(x) for x in proposal_case["V"])
print("2V in E8:", checker.e8_member(tuple(2*x for x in V)))
print("4V in E8:", checker.e8_member(tuple(4*x for x in V)))
print("Weyl adjoint parities:", checker.local_weyl_audit()["adjoint_parities"])
for name in ("local_root_action", "local_projected_dimensions", "declared_shift_order"):
    print("\n" + name)
    print(json.dumps(by_name[name], indent=2))
print("\nAmbient-root-set diagnostic:")
print(json.dumps(evaluated["diagnostics"], indent=2))


2V in E8: False
4V in E8: True
Weyl adjoint parities: [1, -1, -1]

local_root_action
{
  "name": "local_root_action",
  "status": "PASS",
  "evidence": {
    "computed_positive_root_phases_mod_1": [
      "0",
      "0",
      "0",
      "0",
      "0",
      "0",
      "1/2",
      "0",
      "0",
      "0",
      "0",
      "0",
      "0",
      "0",
      "0",
      "0"
    ],
    "target_positive_root_phases_mod_1": [
      "0",
      "0",
      "0",
      "0",
      "0",
      "0",
      "1/2",
      "0",
      "0",
      "0",
      "0",
      "0",
      "0",
      "0",
      "0",
      "0"
    ],
    "order_two_on_selected_roots": true
  },
  "scope": "Abstract A1^8 plus A1^8 subsystem; the second E8 factor is untouched"
}

local_projected_dimensions
{
  "name": "local_projected_dimensions",
  "status": "PASS",
  "evidence": {
    "computed_vector_dimension": 46,
    "computed_scalar_dimensions": [
      46,
      2,
      2
    ],
    "target_vector_dimension": 46,
    "target_s

## Frozen arithmetic and finite reference scan

Use the positive Euclidean pairing for root characters, but the declared negative gauge norm $q_V=-\sum_iV_i^2$ in the following benchmark congruence. For the $U\oplus U$ basis,
$$q_\delta=2xy+2uv,\qquad \phi=(1/2,1/2,0).$$

The implemented condition is
$$N^2(q_V+q_\delta-\phi^2)\in 2N\mathbb Z.$$

With $N=4$ and $\delta=(a/4,b/4;0,1/2)$, this reduces to $ab\equiv1\pmod4$. Exhausting sixteen residue pairs checks the saved proposal against an exact reference. The control uses $V=0$, $N=2$ and the corresponding four residue pairs. Counts are raw arithmetic representatives, not inequivalent physical models.


In [3]:
for case in (baseline, control):
    scan = checker.finite_shift_scan(case["N"], case["V"], case["phi"])
    print(case["id"])
    print("raw pairs:", scan["raw_pair_count"], "survivors:", scan["surviving_pairs"])
    check = next(c for c in checker.evaluate_candidate(case)["checks"]
                 if c["name"] == "level_matching")
    print("level matching:", json.dumps(check["evidence"], sort_keys=True))
    print()
print("Primary scan, all residues:")
for entry in result["reference_scans"][baseline["id"]]["all_pairs"]:
    print((entry["a"], entry["b"]), "residue", entry["residue"],
          "PASS" if entry["passes_congruence"] else "FAIL")


one_higgs_reference
raw pairs: 16 survivors: [[1, 1], [3, 3]]
level matching: {"N2_qV": "-2", "N2_qdelta": "2", "N2_qphi": "8", "congruence_value": "-8", "modulus": 8, "qV": "-1/8", "qdelta": "1/8", "qphi": "1/2", "residue": "0"}

all_coulomb_control
raw pairs: 4 survivors: [[1, 1]]
level matching: {"N2_qV": "0", "N2_qdelta": "2", "N2_qphi": "2", "congruence_value": "0", "modulus": 4, "qV": "0", "qdelta": "1/2", "qphi": "1/2", "residue": "0"}

Primary scan, all residues:
(0, 0) residue 6 FAIL
(0, 1) residue 6 FAIL
(0, 2) residue 6 FAIL
(0, 3) residue 6 FAIL
(1, 0) residue 6 FAIL
(1, 1) residue 0 PASS
(1, 2) residue 2 FAIL
(1, 3) residue 4 FAIL
(2, 0) residue 6 FAIL
(2, 1) residue 2 FAIL
(2, 2) residue 6 FAIL
(2, 3) residue 2 FAIL
(3, 0) residue 6 FAIL
(3, 1) residue 4 FAIL
(3, 2) residue 2 FAIL
(3, 3) residue 0 PASS


## Six fixtures and the review boundary

The suite contains the two reference inputs and four seeded invalid inputs: wrong geometric assignment, wrong declared order, invalid gauge-lattice membership, and failed level matching. A mutation may fail several checks. These fixtures are labelled tests, not spontaneous model mistakes.

A computational rerun should reproduce the saved JSON exactly. `ERROR` denotes malformed data or an execution problem; it is not a demonstrated physical obstruction. `logs/raw_model_response.json` and `logs/review_log.json` preserve the actual model-assisted step, while `HUMAN_REVIEW.md` leaves independent scientific review explicitly pending.

For a complete construction, the massless roots in the actual Narain lattice, including possible toroidal components, must be known. Full phases involve $P\cdot(V+\delta)$. The three unresolved physical checks below cannot be replaced by the arithmetic pass.


In [4]:
print("Fixture | scoped status | overall status | failed checks")
for case in result["cases"]:
    failed = [c["name"] for c in case["checks"] if c["status"] == "FAIL"]
    print(case["id"], case["scoped_status"], case["overall_status"], failed, sep=" | ")
print("\nUnresolved checks for the main example:")
for entry in evaluated["checks"]:
    if entry["status"] == "UNRESOLVED":
        print(entry["name"], "->", entry["evidence"]["required"])
saved = json.loads((ROOT / "results.json").read_text())
assert json.loads(json.dumps(result)) == saved, "Regenerated result differs from the saved record"
assert result["summary"]["verified_duals"] == 0
print("\nExact match with saved results.json. Independent human review: pending.")


Fixture | scoped status | overall status | failed checks
one_higgs_reference | PASS | UNRESOLVED | []
all_coulomb_control | PASS | UNRESOLVED | []
seeded_geometry_mismatch | FAIL | FAIL | ['geometry_target']
seeded_wrong_order | FAIL | FAIL | ['gauge_lattice_membership', 'torus_lattice_membership', 'declared_shift_order', 'level_matching']
seeded_lattice_failure | FAIL | FAIL | ['gauge_lattice_membership', 'declared_shift_order', 'local_root_action', 'local_projected_dimensions', 'level_matching']
seeded_level_matching_failure | FAIL | FAIL | ['level_matching']

Unresolved checks for the main example:
narain_embedding -> Integral Narain basis, Gram form, gluing, moduli/Wilson lines and actual enhanced roots
operator_lift -> Cocycles/phases and consistency of all powers and sectors
complete_spectrum_branch_match -> Full P dot (V+delta), twisted and bulk states, and the branch dictionary

Exact match with saved results.json. Independent human review: pending.
